# 02 — Loading Remaining Datasets
## SBDR Phase A, Step A2
UCI is done (NB01). Now we load the other 4 datasets, do quick health checks, and save processed versions.

In [1]:
!pip install polars

In [2]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports loaded.")

All imports loaded.


## Dataset 1: Lending Club (2.26M loans)
Using Polars for this one — too large for Pandas to handle efficiently. We only load the columns relevant to our project.

In [3]:
# Columns we need for SBDR
lc_cols = [
    'loan_status', 'loan_amnt', 'funded_amnt', 'annual_inc', 
    'dti', 'revol_util', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'purpose',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc',
    'pub_rec', 'total_acc', 'installment', 'int_rate'
]

# Lazy scan — doesn't load into memory until we collect()
lc = pl.scan_csv(
    "../data/raw/LendingClub_accepted2007_2018.csv",
    ignore_errors=True
).select(lc_cols).collect()

print(f"Shape: {lc.shape}")
print(f"Columns: {lc.columns}")
lc.head()

Shape: (2260701, 18)
Columns: ['loan_status', 'loan_amnt', 'funded_amnt', 'annual_inc', 'dti', 'revol_util', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'purpose', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'total_acc', 'installment', 'int_rate']


loan_status,loan_amnt,funded_amnt,annual_inc,dti,revol_util,grade,sub_grade,emp_length,home_ownership,purpose,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,total_acc,installment,int_rate
str,f64,f64,f64,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64
"""Fully Paid""",3600.0,3600.0,55000.0,5.91,29.7,"""C""","""C4""","""10+ years""","""MORTGAGE""","""debt_consolidation""",0.0,1.0,7.0,0.0,13.0,123.03,13.99
"""Fully Paid""",24700.0,24700.0,65000.0,16.06,19.2,"""C""","""C1""","""10+ years""","""MORTGAGE""","""small_business""",1.0,4.0,22.0,0.0,38.0,820.28,11.99
"""Fully Paid""",20000.0,20000.0,63000.0,10.78,56.2,"""B""","""B4""","""10+ years""","""MORTGAGE""","""home_improvement""",0.0,0.0,6.0,0.0,18.0,432.66,10.78
"""Current""",35000.0,35000.0,110000.0,17.06,11.6,"""C""","""C5""","""10+ years""","""MORTGAGE""","""debt_consolidation""",0.0,0.0,13.0,0.0,17.0,829.9,14.85
"""Fully Paid""",10400.0,10400.0,104433.0,25.37,64.5,"""F""","""F1""","""3 years""","""MORTGAGE""","""major_purchase""",1.0,3.0,12.0,0.0,35.0,289.91,22.45


In [4]:
# Data types
print("=== Data Types ===")
print(lc.dtypes)
print(f"\n=== Null Counts ===")
print(lc.null_count())

=== Data Types ===
[String, Float64, Float64, Float64, Float64, Float64, String, String, String, String, String, Float64, Float64, Float64, Float64, Float64, Float64, Float64]

=== Null Counts ===
shape: (1, 18)
┌────────────┬───────────┬────────────┬───────────┬───┬─────────┬───────────┬───────────┬──────────┐
│ loan_statu ┆ loan_amnt ┆ funded_amn ┆ annual_in ┆ … ┆ pub_rec ┆ total_acc ┆ installme ┆ int_rate │
│ s          ┆ ---       ┆ t          ┆ c         ┆   ┆ ---     ┆ ---       ┆ nt        ┆ ---      │
│ ---        ┆ u32       ┆ ---        ┆ ---       ┆   ┆ u32     ┆ u32       ┆ ---       ┆ u32      │
│ u32        ┆           ┆ u32        ┆ u32       ┆   ┆         ┆           ┆ u32       ┆          │
╞════════════╪═══════════╪════════════╪═══════════╪═══╪═════════╪═══════════╪═══════════╪══════════╡
│ 33         ┆ 33        ┆ 33         ┆ 37        ┆ … ┆ 62      ┆ 62        ┆ 33        ┆ 33       │
└────────────┴───────────┴────────────┴───────────┴───┴─────────┴───────────┴────

In [5]:
# What loan outcomes exist?
status_counts = lc.group_by('loan_status').agg(
    pl.len().alias('count')
).sort('count', descending=True)

print(status_counts)

shape: (10, 2)
┌─────────────────────────────────┬─────────┐
│ loan_status                     ┆ count   │
│ ---                             ┆ ---     │
│ str                             ┆ u32     │
╞═════════════════════════════════╪═════════╡
│ Fully Paid                      ┆ 1076751 │
│ Current                         ┆ 878317  │
│ Charged Off                     ┆ 268559  │
│ Late (31-120 days)              ┆ 21467   │
│ In Grace Period                 ┆ 8436    │
│ Late (16-30 days)               ┆ 4349    │
│ Does not meet the credit polic… ┆ 1988    │
│ Does not meet the credit polic… ┆ 761     │
│ Default                         ┆ 40      │
│ null                            ┆ 33      │
└─────────────────────────────────┴─────────┘


### Loan Status → SBDR Tier Mapping
We map Lending Club's loan outcomes to our 4-tier recovery system. This gives us real-world labels for how customers progress through financial distress.

In [6]:
# Map loan status to SBDR tiers
tier_map = {
    'Fully Paid': 'No Action',
    'Current': 'No Action',
    'In Grace Period': 'Tier 1',
    'Late (16-30 days)': 'Tier 2',
    'Late (31-120 days)': 'Tier 3',
    'Charged Off': 'Tier 4',
    'Default': 'Tier 4'
}

# Apply mapping, drop rows that don't fit
lc = lc.with_columns(
    pl.col('loan_status').replace(tier_map).alias('sbdr_tier')
).filter(
    pl.col('sbdr_tier').is_in(['No Action', 'Tier 1', 'Tier 2', 'Tier 3', 'Tier 4'])
)

# Check distribution
print(lc.group_by('sbdr_tier').agg(
    pl.len().alias('count')
).sort('count', descending=True))

print(f"\nCleaned shape: {lc.shape}")

shape: (5, 2)
┌───────────┬─────────┐
│ sbdr_tier ┆ count   │
│ ---       ┆ ---     │
│ str       ┆ u32     │
╞═══════════╪═════════╡
│ No Action ┆ 1955068 │
│ Tier 4    ┆ 268599  │
│ Tier 3    ┆ 21467   │
│ Tier 1    ┆ 8436    │
│ Tier 2    ┆ 4349    │
└───────────┴─────────┘

Cleaned shape: (2257919, 19)


Clean. Lost only ~2,800 rows:

    No Action (1.95M) — most loans are fine
    Tier 4 (269K) — bank already gave up on these
    Tiers 1-3 (34K combined) — the customers SBDR is built to help

In [7]:
# Converting to Pandas for saving (Polars did its job — heavy lifting is done)
lc_pd = lc.to_pandas()

lc_pd.to_csv("../data/processed/lending_club_processed.csv", index=False)
print(f"Saved: {lc_pd.shape[0]:,} rows × {lc_pd.shape[1]} columns")
print("→ data/processed/lending_club_processed.csv")

Saved: 2,257,919 rows × 19 columns
→ data/processed/lending_club_processed.csv


### Lending Club — Done ✅
- **Raw:** 2.26M rows × 151 columns
- **Processed:** ~2.26M rows × 19 columns (18 selected + sbdr_tier)
- **Key insight:** 34K customers in Tiers 1–3 — the "saveable" group our system targets
- Nulls negligible (~0.003%), dropped with tier mapping

---
## Dataset 2: Sparkov Synthetic Transactions
Combining train + test into one pool. These give us daily spending profiles to detect anomalies.

In [8]:
# Loading both files and combine
spk_train = pd.read_csv("../data/raw/Sparkov Synthetic/fraudTrain.csv")
spk_test = pd.read_csv("../data/raw/Sparkov Synthetic/fraudTest.csv")

spk = pd.concat([spk_train, spk_test], ignore_index=True)

print(f"Train: {spk_train.shape}")
print(f"Test:  {spk_test.shape}")
print(f"Combined: {spk.shape}")
print(f"\nColumns: {list(spk.columns)}")

Train: (1296675, 23)
Test:  (555719, 23)
Combined: (1852394, 23)

Columns: ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


In [9]:
print(f"=== Null Values ===")
print(spk.isnull().sum()[spk.isnull().sum() > 0])
if spk.isnull().sum().sum() == 0:
    print("No nulls!")

print(f"\n=== Target: is_fraud ===")
fraud_counts = spk['is_fraud'].value_counts()
print(f"Not Fraud (0): {fraud_counts[0]:,} ({fraud_counts[0]/len(spk)*100:.2f}%)")
print(f"Fraud (1):     {fraud_counts[1]:,} ({fraud_counts[1]/len(spk)*100:.2f}%)")

print(f"\n=== Sample Categories ===")
print(spk['category'].value_counts().head(10))

=== Null Values ===
Series([], dtype: int64)
No nulls!

=== Target: is_fraud ===
Not Fraud (0): 1,842,743 (99.48%)
Fraud (1):     9,651 (0.52%)

=== Sample Categories ===
category
gas_transport     188029
grocery_pos       176191
home              175460
shopping_pos      166463
kids_pets         161727
shopping_net      139322
entertainment     134118
food_dining       130729
personal_care     130085
health_fitness    122553
Name: count, dtype: int64


Zero nulls, perfect. The categories are— health_fitness, food_dining, gas_transport etc. When someone's medical spending spikes or grocery spending drops, that's a "Life Event" signal for our BiLSTM.

In [10]:
# Columns we need for SBDR
spk_cols = [
    'cc_num',                  # customer identifier
    'trans_date_trans_time',   # when
    'category',                # what they spent on
    'amt',                     # how much
    'gender', 'dob',           # demographics
    'city', 'state',           # location
    'is_fraud'                 # fraud flag
]

spk_clean = spk[spk_cols].copy()

# Rename for clarity
spk_clean.rename(columns={
    'cc_num': 'customer_id',
    'trans_date_trans_time': 'transaction_date',
    'amt': 'amount'
}, inplace=True)

print(f"Cleaned shape: {spk_clean.shape}")
print(f"\nUnique customers: {spk_clean['customer_id'].nunique():,}")
print(f"Date range: {spk_clean['transaction_date'].min()} → {spk_clean['transaction_date'].max()}")

spk_clean.to_csv("../data/processed/sparkov_processed.csv", index=False)
print(f"\nSaved → data/processed/sparkov_processed.csv")

Cleaned shape: (1852394, 9)

Unique customers: 999
Date range: 2019-01-01 00:00:18 → 2020-12-31 23:59:34

Saved → data/processed/sparkov_processed.csv


### Sparkov Synthetic — Done ✅
- **Raw:** 1.85M transactions (train + test combined)
- **Processed:** 9 columns kept — customer ID, date, category, amount, demographics, fraud flag
- **No nulls**, fraud rate 0.52%
- Key categories for Life Event detection: health_fitness, food_dining, gas_transport
- Will be used to build daily/monthly spending profiles per customer

---
## Dataset 3: Financial PhraseBank (HuggingFace)
Loading directly from HuggingFace — no file download needed. This trains our FinBERT to understand financial sentiment.

In [14]:
# Load Financial PhraseBank — AllAgree (highest quality labels)
fpb_lines = open(
    "../data/raw/FinancialPhraseBank-v1.0/Sentences_AllAgree.txt", 
    encoding="latin-1"
).readlines()

# Each line: "sentence@label"
data = []
for line in fpb_lines:
    parts = line.strip().rsplit("@", 1)
    if len(parts) == 2:
        data.append({'sentence': parts[0].strip(), 'label': parts[1].strip()})

fpb_df = pd.DataFrame(data)

print(f"Shape: {fpb_df.shape}")
print(f"\n=== Sentiment Distribution ===")
print(fpb_df['label'].value_counts())
fpb_df.head()

Shape: (2264, 2)

=== Sentiment Distribution ===
label
neutral     1391
positive     570
negative     303
Name: count, dtype: int64


,sentence,label
0,"According to Gran , the company has no plans t...",neutral
1,"For the last quarter of 2010 , Componenta 's n...",positive
2,"In the third quarter of 2010 , net sales incre...",positive
3,Operating profit rose to EUR 13.1 mn from EUR ...,positive
4,"Operating profit totalled EUR 21.1 mn , up fro...",positive


In [15]:
fpb_df.to_csv("../data/processed/financial_phrasebank_processed.csv", index=False)
print(f"Saved: {fpb_df.shape[0]:,} rows × {fpb_df.shape[1]} columns")
print("→ data/processed/financial_phrasebank_processed.csv")

Saved: 2,264 rows × 2 columns
→ data/processed/financial_phrasebank_processed.csv


### Financial PhraseBank — Done ✅
- **Source:** Sentences_AllAgree.txt (all 16 annotators agreed)
- **Shape:** 2,264 sentences × 2 columns (sentence + label)
- **Labels:** neutral (61%), positive (25%), negative (14%)
- Will be used to fine-tune FinBERT for financial sentiment detection

---
## Dataset 4: Mental Health Sentiment
This teaches our NLP model to detect anxiety, stress, and depression language — the emotional layer that Financial PhraseBank doesn't cover.

In [16]:
mh = pd.read_csv("../data/raw/Sentiment Analysis for Mental Health.csv")

print(f"Shape: {mh.shape}")
print(f"Columns: {list(mh.columns)}")
print(f"\nNulls:\n{mh.isnull().sum()}")
print(f"\n=== Sentiment/Label Distribution ===")
print(mh[mh.columns[-1]].value_counts())
mh.head()

Shape: (53043, 3)
Columns: ['Unnamed: 0', 'statement', 'status']

Nulls:
Unnamed: 0      0
statement     362
status          0
dtype: int64

=== Sentiment/Label Distribution ===
status
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64


,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety


In [17]:
# Drop nulls and useless index column
mh_clean = mh.drop(columns=['Unnamed: 0']).dropna(subset=['statement'])

# Keep only categories relevant to financial distress
relevant_statuses = ['Anxiety', 'Stress', 'Depression', 'Normal']
mh_clean = mh_clean[mh_clean['status'].isin(relevant_statuses)].copy()

print(f"Cleaned shape: {mh_clean.shape}")
print(f"\n=== Filtered Distribution ===")
print(mh_clean['status'].value_counts())

mh_clean.to_csv("../data/processed/mental_health_processed.csv", index=False)
print(f"\nSaved → data/processed/mental_health_processed.csv")

Cleaned shape: (38175, 2)

=== Filtered Distribution ===
status
Normal        16343
Depression    15404
Anxiety        3841
Stress         2587
Name: count, dtype: int64

Saved → data/processed/mental_health_processed.csv


### Mental Health Sentiment — Done ✅
- **Raw:** 53,043 rows × 7 categories
- **Processed:** ~38K rows × 4 relevant categories (Normal, Depression, Anxiety, Stress)
- Dropped: Suicidal, Bipolar, Personality disorder (outside SBDR scope)
- 362 null statements dropped
- Will teach FinBERT to detect emotional distress language in customer communications